# Wynn's World
### A Cognitive Architecture Confronts Counting Dolls

Companion notebook to the paper of the same name.  
Sections §3–§7 reproduce the findings reported there, in paper order. §DC is the director's cut.

**Requirements:** Python 3.10+, PyTorch. No GPU needed.

In [ ]:
import os, sys, subprocess
from pathlib import Path
from IPython.display import Markdown, display

# ── locate repo ───────────────────────────────────────────────────────────
if Path('/content').exists():          # Google Colab
    REPO_URL = 'https://github.com/kubasiemion/emergent_ontology'
    ROOT = Path('/content/emergent_ontology')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
else:                                  # local
    ROOT = Path.cwd()
    while not (ROOT / 'arena').exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

ARENA   = ROOT / 'arena'
MATCHES = ARENA / 'matches'
SRC     = ROOT / 'src'
sys.path.insert(0, str(SRC))
os.chdir(ROOT)
print(f'root: {ROOT}')

In [2]:
# ── helpers ───────────────────────────────────────────────────────────────────
_env = os.environ.copy()
_env['PYTHONPATH'] = str(SRC)

def run(match_name):
    """Run a match and display results."""
    r = subprocess.run(
        [sys.executable, str(ARENA / 'run_match.py'), str(MATCHES / match_name)],
        env=_env, capture_output=True, text=True, cwd=str(ROOT),
    )
    if r.returncode != 0:
        print(r.stderr)
        return
    show(match_name)

def show(match_name):
    """Display committed results without re-running."""
    display(Markdown((MATCHES / match_name / 'results.md').read_text()))

In [ ]:
# ── mode ─────────────────────────────────────────────────────────────────────
# Set RECOMPUTE = True to re-run all matches from scratch.
# False (default): display committed results instantly — no compute needed.
RECOMPUTE = False

def exhibit(match_name):
    run(match_name) if RECOMPUTE else show(match_name)

## Glossary

| Term | Meaning |
|---|---|
| **deficit** | `max(1/n_tokens − pred_prob[next], 0)` summed over steps, averaged over sequences. Zero = perfect prediction. Higher = worse. |
| **wiring** | A mapping from perception tokens (INC, DEC, C0…) to model tokens. The matcher searches for the wiring that minimises deficit. |
| **ghost** | A model with the architecture of a larger world but trained only to a smaller boundary. The extra slots exist; the representations do not. |
| **canonical wiring** | The identity mapping: INC→INC, DEC→DEC, C0→C0, … The correct wiring when model and world share the same token convention. |
| **contender set** | Wirings retained after cliff pruning for extension to the next stage. The hedge against a future the current world does not reveal. |
| **cliff pruning** | Keep all wirings below the largest gap in the sorted deficit list. |
| **succession** | The rank crossing between stages: the stage-1 leader becomes the stage-2 loser. |

---
## §3 · Matching and Underdetermination

The matcher enumerates all ways to connect perception tokens to model tokens and ranks them by predictive deficit.  
On the cap3 world (three dolls, vocabulary {INC, DEC, C0, C1, C2}) it finds two valid wiring families — canonical and reverse-canonical — across all models.  
The underdetermination was not designed in. The counting world has a reflection symmetry, and the matcher finds it.

In [3]:
exhibit('cap3_ghost_games')

# Cap3 world: native cap3 models vs cap4 (fully trained) vs cap4 ghost (trained only to cap3).

| Place |  Deficit | Model                         | INC | DEC | C0  | C1  | C2  |
| ----- | -------- | ----------------------------- | --- | --- | --- | --- | --- |
|     1 |   0.0000 | cap4_ghost | INC | DEC | C0  | C1  | C2  |
|     2 |   0.0102 | cap3_acc70     | INC | DEC | C0  | C1  | C2  |
|     3 |   0.0136 | cap4_ghost | DEC | INC | C2  | C1  | C0  |
|     4 |   0.0287 | cap3_acc80     | DEC | INC | C2  | C1  | C0  |
|     5 |   0.0573 | cap4_acc80     | DEC | INC | C2  | C1  | C0  |
|     6 |   0.0712 | cap3_acc70     | DEC | INC | C2  | C1  | C0  |
|     7 |   0.0789 | cap3_acc80     | INC | DEC | C0  | C1  | C2  |
|     8 |   0.2405 | cap4_acc80     | INC | DEC | C0  | C1  | C2  |
|     9 |   0.3025 | cap4_acc80     | INC | DEC | C0  | C1  | C3  |
|    10 |   0.3857 | cap4_acc80     | DEC | INC | C3  | C1  | C0  |
|    11 |   1.5831 | cap3_acc70     | C1  | DEC | C0  | INC | C2  |
|    12 |   1.6048 | cap4_acc80     | INC | DEC | C1  | C2  | C3  |
|    13 |   1.6596 | cap3_acc70     | DEC | C1  | C2  | INC | C0  |
|    14 |   1.6852 | cap4_acc80     | INC | DEC | C0  | C2  | C3  |
|    15 |   1.7574 | cap3_acc70     | C1  | INC | C2  | DEC | C0  |
|    16 |   1.8209 | cap3_acc70     | INC | C1  | C0  | DEC | C2  |
|    17 |   1.8999 | cap4_ghost | C1  | DEC | C0  | INC | C2  |
|    18 |   1.9009 | cap4_ghost | DEC | C1  | C2  | INC | C0  |
|    19 |   1.9583 | cap4_ghost | INC | C1  | C0  | DEC | C2  |
|    20 |   2.0370 | cap4_ghost | C1  | INC | C2  | DEC | C0  |
|    21 |   2.6507 | cap3_acc80     | DEC | C1  | C2  | INC | C0  |
|    22 |   2.6670 | cap3_acc80     | INC | C1  | C0  | DEC | C2  |
|    23 |   2.9658 | cap3_acc80     | C1  | INC | C2  | DEC | C0  |
|    24 |   3.0210 | cap3_acc80     | C1  | DEC | C0  | INC | C2  |


**Reading the result.**  
The top two wirings are the canonical family (INC→INC, DEC→DEC, C0→C0…) and the reverse-canonical family (INC→DEC, DEC→INC, C0→C2…). Both score near zero. The architecture was not told which way is 'up' — the counting world has a reflection symmetry, and the matcher finds both valid orientations. This is underdetermination encountered, not stipulated: it falls out of the deficit landscape.

---
## §4 · A New Doll Arrives: The Ghost Experiment

The **ghost** is a model with the architecture of a larger world but trained only to a smaller boundary.  
A cap4 ghost trained to cap3 has the C3 slot in its vocabulary — but C3 received zero gradient. The slot exists. The representation does not.

On the home world (cap3) the ghost wins: deficit 0.000 on canonical wiring.  
On the extended world (cap4) the ghost collapses: deficit 7.07, gap 7.03 against the native model.

The second match decomposes *where* the deficit lands.

In [4]:
exhibit('cap4_ghost_games')

# Cap4 world: fully-trained cap4 vs cap4-ghost (trained only to cap3) vs symbolic oracle.

| Place |  Deficit | Model                         | INC | DEC | C0  | C1  | C2  | C3  |
| ----- | -------- | ----------------------------- | --- | --- | --- | --- | --- | --- |
|     1 |   0.0000 | symbolic_cap4                 | INC | DEC | C0  | C1  | C2  | C3  |
|     2 |   0.0000 | symbolic_cap4                 | DEC | INC | C3  | C2  | C1  | C0  |
|     3 |   0.0400 | cap4_acc80     | INC | DEC | C0  | C1  | C2  | C3  |
|     4 |   0.9823 | cap4_acc80     | DEC | INC | C3  | C2  | C1  | C0  |
|     5 |   3.1905 | cap4_acc80     | INC | DEC | C0  | C1  | C3  | C2  |
|     6 |   3.3789 | cap4_acc80     | INC | DEC | C0  | C2  | C1  | C3  |
|     7 |   4.0405 | cap4_acc80     | DEC | INC | C3  | C1  | C2  | C0  |
|     8 |   4.3333 | symbolic_cap4                 | INC | DEC | C1  | C2  | C3  | C0  |
|     9 |   4.3333 | symbolic_cap4                 | DEC | INC | C2  | C1  | C0  | C3  |
|    10 |   4.7333 | symbolic_cap4                 | INC | DEC | C3  | C0  | C1  | C2  |
|    11 |   4.7333 | symbolic_cap4                 | DEC | INC | C0  | C3  | C2  | C1  |
|    12 |   4.8028 | cap4_acc80     | DEC | INC | C2  | C3  | C1  | C0  |
|    13 |   6.3606 | cap4_ghost | INC | C1  | C0  | DEC | C2  | C3  |
|    14 |   6.4881 | cap4_ghost | DEC | C1  | C2  | INC | C0  | C3  |
|    15 |   7.0341 | cap4_ghost | C1  | DEC | C3  | C0  | INC | C2  |
|    16 |   7.0694 | cap4_ghost | INC | DEC | C0  | C1  | C2  | C3  |
|    17 |   7.1203 | cap4_ghost | C1  | INC | C3  | C2  | DEC | C0  |
|    18 |   7.6556 | cap4_ghost | DEC | INC | C3  | C1  | C2  | C0  |


In [5]:
# Deficit decomposition by predicted token type.
# Scenario A (world extension) is the focus here; Scenario B is §5.
exhibit('cap4_error_structure')

# Scenario A — World Extension  (cap4 world, canonical wiring, new token = C3)

| Model                         |   INC   |   DEC   |   C0    |   C1    |   C2    |   C3    |    Total |
| ----------------------------- | ------- | ------- | ------- | ------- | ------- | ------- | -------- |
| cap4_ghost |  0.0000 |  0.0000 |  0.0000 |  0.0300 |  3.3545 |  3.6849 |   7.0694 |
|   /step                       |  0.0000 |  0.0000 |  0.0000 |  0.0001 |  0.0120 |  0.0297 |          |
| cap4_acc80     |  0.0000 |  0.0000 |  0.0000 |  0.0400 |  0.0000 |  0.0000 |   0.0400 |
|   /step                       |  0.0000 |  0.0000 |  0.0000 |  0.0001 |  0.0000 |  0.0000 |          |

# Scenario B — Screen Violation  (cap3 world, 10% count corruption)

| Model                         |   INC   |   DEC   |   C0    |   C1    |   C2    |    Total |
| ----------------------------- | ------- | ------- | ------- | ------- | ------- | -------- |
| cap4_ghost |  0.0000 |  0.0000 |  0.5338 |  0.0095 |  0.5704 |   1.1137 |
|   /step                       |  0.0000 |  0.0000 |  0.0023 |  0.0000 |  0.0027 |          |
| cap4_acc80     |  0.0000 |  0.0000 |  1.0890 |  0.1598 |  0.4450 |   1.6938 |
|   /step                       |  0.0000 |  0.0000 |  0.0048 |  0.0003 |  0.0021 |          |

## Discrimination gap (max − min across models, per token)

Scenario A:  INC=0.0000  DEC=0.0000  C0=0.0000  C1=0.0100  C2=3.3545  C3=3.6849
Scenario B:  INC=0.0000  DEC=0.0000  C0=0.5552  C1=0.1503  C2=0.1254

Total gap — A: 7.0494   B: 0.8310


**Reading the decomposition.**  
98.9% of the ghost's deficit concentrates at C2 and C3.  
C3 is direct failure — no representation for count=3.  
C2 is boundary contamination — hidden state corrupted after the ghost's empty C3 slot cascades into adjacent transitions (count 3→2 via DEC).  
The per-step deficit at C3 is 0.030 vs 0.000 for the native model.  
The decomposition is a map of the learning problem: the crisis is located exactly at the training boundary.

---
## §5 · When the World Cheats: Two Kinds of Signal

Two things can go wrong. A new doll can appear (world extension). Or the experimenter can cheat — a doll is placed in the wrong position (screen violation).  
Both produce prediction error. The architecture distinguishes them.

The violation results (Scenario B in the `cap4_error_structure` table above, cap3 + 10% count corruption) show the ghost *outperforming* the native: gap 0.58, inverted.  
Acting on the violation signal would promote the ghost — the wrong move for world expansion.

In [6]:
# Replication at cap5 scale.
# Ghost trained to cap4; new token C4; violation on cap4 sequences.
exhibit('cap5_error_structure')

# Scenario A — World Extension  (cap5 world, canonical wiring, new token = C4)

| Model                         |   INC   |   DEC   |   C0    |   C1    |   C2    |   C3    |   C4    |    Total |
| ----------------------------- | ------- | ------- | ------- | ------- | ------- | ------- | ------- | -------- |
| cap5_cap4trained_acc80 |  0.0000 |  0.0000 |  0.0166 |  0.0000 |  0.1876 |  0.4667 |  2.2179 |   2.8887 |
|   /step                       |  0.0000 |  0.0000 |  0.0001 |  0.0000 |  0.0009 |  0.0022 |  0.0258 |          |
| cap5_acc80     |  0.0000 |  0.0000 |  0.0391 |  0.0000 |  0.0000 |  0.0000 |  0.0000 |   0.0391 |
|   /step                       |  0.0000 |  0.0000 |  0.0004 |  0.0000 |  0.0000 |  0.0000 |  0.0000 |          |

# Scenario B — Screen Violation  (cap4 world, 10% count corruption)

| Model                         |   INC   |   DEC   |   C0    |   C1    |   C2    |   C3    |    Total |
| ----------------------------- | ------- | ------- | ------- | ------- | ------- | ------- | -------- |
| cap5_cap4trained_acc80 |  0.0000 |  0.0000 |  0.4055 |  0.0657 |  0.1536 |  0.3333 |   0.9581 |
|   /step                       |  0.0000 |  0.0000 |  0.0027 |  0.0002 |  0.0006 |  0.0025 |          |
| cap5_acc80     |  0.0000 |  0.0000 |  1.0082 |  0.3533 |  0.1110 |  0.1781 |   1.6505 |
|   /step                       |  0.0000 |  0.0000 |  0.0068 |  0.0010 |  0.0004 |  0.0013 |          |

## Discrimination gap (max − min across models, per token)

Scenario A:  INC=0.0000  DEC=0.0000  C0=0.0225  C1=0.0000  C2=0.1876  C3=0.4667  C4=2.2179
Scenario B:  INC=0.0000  DEC=0.0000  C0=0.6027  C1=0.2876  C2=0.0427  C3=0.1552

Total gap — A: 2.8946   B: 1.0882


| Signal | Cap4 gap | Cap5 gap | Direction |
|---|---|---|---|
| World extension | 7.03 | 2.85 | Correct — native wins |
| Screen violation | 0.58 | 0.69 | Inverted — ghost wins |

The presence or absence of a new world token is not incidental. It is what makes the discrimination signal point in the right direction.

---
## §6 · The Itch Before the Evidence: Lateral Detection

Can the system detect model inadequacy before the new token explicitly appears?

The match runs two passes. **Pass 1 (filtered):** all C3 tokens are removed from the evaluation stream — 8.3% of events dropped. The model never sees the new token. **Pass 2 (full):** complete cap4 stream, included as a comparison baseline.

The key result is in Pass 1.

In [ ]:
exhibit('cap4_lateral_detection')

**Reading the result.**  
Pass 1 (filtered): ghost deficit 3.38, native 0.04. The ghost fails before C3 ever appears — the boundary dynamics in INC/DEC are already wrong.  
Pass 2 (full stream): same ranking, higher ghost deficit. Shown for comparison only — this replicates the §4 ghost result on the full cap4 world.

---
## §7 · The Crossing: Succession as Architectural Event

Incremental wiring: the matcher scores models on the home world (stage 1), retains contenders, then extends surviving wirings to the expanded world (stage 2).  
The stage-1 leader becomes the stage-2 loser. The crossing between stages is the succession event — inspectable as a rank reversal in deficit tables.

In [8]:
# Cap4 succession: 2-stage, ghost vs native.
exhibit('cap4_succession')

# Stage 1 — cap3 departure  (cliff pruning)

Contenders surviving to stage 2: **6**
- **cap4_ghost**: 2
- **cap4_acc80**: 4

| Place |  Deficit | Model                         | INC | DEC | C0  | C1  | C2  |
| ----- | -------- | ----------------------------- | --- | --- | --- | --- | --- |
|     1 |   0.0000 | cap4_ghost | INC | DEC | C0  | C1  | C2  |
|     2 |   0.0136 | cap4_ghost | DEC | INC | C2  | C1  | C0  |
|     3 |   0.0577 | cap4_acc80     | DEC | INC | C2  | C1  | C0  |
|     4 |   0.3795 | cap4_acc80     | INC | DEC | C0  | C1  | C2  |

# Stage 2 — cap4 extension  (S1 Base = stage-1 deficit)

| Place |  Deficit |  S1 Base | Model                         | INC | DEC | C0  | C1  | C2  | C3  |
| ----- | -------- | -------- | ----------------------------- | --- | --- | --- | --- | --- | --- |
|     1 |   0.0400 |   0.3795 | cap4_acc80     | INC | DEC | C0  | C1  | C2  | C3  |
|     2 |   3.1905 |   0.5072 | cap4_acc80     | INC | DEC | C0  | C1  | C3  | C2  |
|     3 |   7.0694 |   0.0000 | cap4_ghost | INC | DEC | C0  | C1  | C2  | C3  |
|     4 |   8.6570 |   0.0136 | cap4_ghost | DEC | INC | C2  | C1  | C0  | C3  |


**Reading the crossing.**

| | Stage 1 deficit | Stage 2 deficit |
|---|---|---|
| ghost (canonical) | **0.0000** ← leader | 7.07 |
| native (canonical) | 0.3795 | **0.0400** ← leader |

The ghost leads stage 1 by a wide margin. The native's winning wiring was rank 4 at stage 1 (deficit 0.3795). At stage 2 the positions invert completely — ghost 7.07, native 0.04. Committing to the stage-1 winner would have produced deficit 10.31 at stage 2.  
The contender set is the hedge against a future the current world has not yet revealed.

In [9]:
# Cap5 succession: 3-stage (cap3 → cap4 → cap5).
# Ghost trained to cap4; competitive through stage 2; collapses at stage 3.
exhibit('cap5_incremental')

# Stage 1 — cap3 departure  (cliff pruning)

Contenders surviving: **9**
- **cap5_acc80**: 2
- **cap5_cap4trained_acc80**: 6
- **cap5_acc70**: 1

| Place |  Deficit | Model                         | INC | DEC | C0  | C1  | C2  |
| ----- | -------- | ----------------------------- | --- | --- | --- | --- | --- |
|     1 |   0.0096 | cap5_acc80     | DEC | INC | C2  | C1  | C0  |
|     2 |   0.0146 | cap5_cap4trained_acc80 | DEC | INC | C2  | C1  | C0  |
|     3 |   0.0166 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C2  |
|     4 |   0.0212 | cap5_acc80     | INC | DEC | C0  | C1  | C2  |
|     5 |   0.0481 | cap5_acc70     | DEC | INC | C2  | C1  | C0  |
|     6 |   0.3065 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C3  |

# Stage 2 — cap4 extension  (cliff pruning)

Contenders surviving: **14**
- **cap5_cap4trained_acc80**: 12
- **cap5_acc80**: 2

| Place |  Deficit |  S1 Base | Model                         | INC | DEC | C0  | C1  | C2  | C3  |
| ----- | -------- | -------- | ----------------------------- | --- | --- | --- | --- | --- | --- |
|     1 |   0.0166 |   0.0166 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C2  | C3  |
|     2 |   0.5457 |   0.0212 | cap5_acc80     | INC | DEC | C0  | C1  | C2  | C3  |
|     3 |   0.5928 |   0.0212 | cap5_acc80     | INC | DEC | C0  | C1  | C2  | C4  |
|     4 |   1.5726 |   0.3065 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C3  | C2  |
|     5 |   3.2518 |   0.0166 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C2  | C4  |
|     6 |   4.5673 |   0.7326 | cap5_cap4trained_acc80 | DEC | INC | C3  | C1  | C0  | C2  |

# Stage 3 — cap5 final  (Prev = stage-2 deficit)

| Place |  Deficit |  S1 Base | Model                         | INC | DEC | C0  | C1  | C2  | C3  | C4  |
| ----- | -------- | -------- | ----------------------------- | --- | --- | --- | --- | --- | --- | --- |
|     1 |   0.0391 |   0.5457 | cap5_acc80     | INC | DEC | C0  | C1  | C2  | C3  | C4  |
|     2 |   0.5813 |   0.5928 | cap5_acc80     | INC | DEC | C0  | C1  | C2  | C4  | C3  |
|     3 |   2.8887 |   0.0166 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C2  | C3  | C4  |
|     4 |   5.0593 |   1.5726 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C3  | C2  | C4  |
|     5 |   5.9592 |   3.2518 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C2  | C4  | C3  |
|     6 |   8.5965 |   4.8078 | cap5_cap4trained_acc80 | INC | DEC | C0  | C1  | C3  | C4  | C2  |


---
---
## §DC · Director's Cut

Six additional findings documented in Appendix A of the companion paper.  
Implementation-level validations rather than architectural exhibits.

### DC-1 · The Interface Principle: GRU Bypass as Functional Requirement

Same model weights, two inference modes: `bypass_read_tokens=True` vs `False`.  
When read tokens enter the GRU, hidden state becomes indistinguishable across count states.  
The matcher cannot recover a coherent wiring. Best deficit without bypass: 6.03 on a scrambled assignment.  
The bypass is not a design preference — it is a functional requirement.

In [10]:
exhibit('cap4_bypass_compare')

# Cap4 acc80 model vs itself: read-token GRU bypass on vs off.

| Place |  Deficit | Model           | INC | DEC | C0  | C1  | C2  | C3  |
| ----- | -------- | --------------- | --- | --- | --- | --- | --- | --- |
|     1 |   0.0518 | cap4_bypass_on  | INC | DEC | C0  | C1  | C2  | C3  |
|     2 |   0.6736 | cap4_bypass_on  | DEC | INC | C3  | C2  | C1  | C0  |
|     3 |   2.8308 | cap4_bypass_on  | INC | DEC | C0  | C2  | C1  | C3  |
|     4 |   3.4324 | cap4_bypass_on  | DEC | INC | C2  | C3  | C1  | C0  |
|     5 |   4.0815 | cap4_bypass_on  | INC | DEC | C0  | C1  | C3  | C2  |
|     6 |   4.4219 | cap4_bypass_on  | DEC | INC | C3  | C1  | C2  | C0  |
|     7 |   6.0289 | cap4_bypass_off | INC | C3  | C0  | DEC | C1  | C2  |
|     8 |   6.8781 | cap4_bypass_off | C3  | INC | C2  | C1  | DEC | C0  |
|     9 |   6.9376 | cap4_bypass_off | INC | C3  | DEC | C0  | C1  | C2  |
|    10 |   7.2878 | cap4_bypass_off | INC | DEC | C3  | C0  | C1  | C2  |
|    11 |   7.3098 | cap4_bypass_off | DEC | INC | C2  | C1  | C0  | C3  |
|    12 |   8.0172 | cap4_bypass_off | C3  | INC | C2  | C1  | C0  | DEC |


### DC-2 · Repeg Degrades Monotonically

Repeg = runtime hidden-state correction toward learned count attractors during matching.  
Hypothesis: for clean curriculum-trained models with GRU bypass active, this should hurt.  
Four alpha values tested. Degradation is strictly monotonic. The symbolic counter (deficit = 0) serves as reference ceiling.

In [11]:
exhibit('cap4_repeg_sweep')

# Cap4 linear — acc80 model at repeg_alpha 0, 0.1, 0.2, 0.5 vs symbolic oracle.

| Place |  Deficit | Model               | INC | DEC | C0  | C1  | C2  | C3  |
| ----- | -------- | ------------------- | --- | --- | --- | --- | --- | --- |
|     1 |   0.0000 | symbolic_cap4       | INC | DEC | C0  | C1  | C2  | C3  |
|     2 |   0.0000 | symbolic_cap4       | DEC | INC | C3  | C2  | C1  | C0  |
|     3 |   0.0518 | cap4_acc80_repeg0.0 | INC | DEC | C0  | C1  | C2  | C3  |
|     4 |   0.6736 | cap4_acc80_repeg0.0 | DEC | INC | C3  | C2  | C1  | C0  |
|     5 |   1.3826 | cap4_acc80_repeg0.1 | INC | DEC | C0  | C1  | C2  | C3  |
|     6 |   1.7068 | cap4_acc80_repeg0.1 | DEC | INC | C3  | C2  | C1  | C0  |
|     7 |   2.0997 | cap4_acc80_repeg0.2 | DEC | INC | C3  | C2  | C1  | C0  |
|     8 |   2.2703 | cap4_acc80_repeg0.2 | INC | DEC | C0  | C1  | C2  | C3  |
|     9 |   2.6268 | cap4_acc80_repeg0.5 | DEC | INC | C3  | C2  | C1  | C0  |
|    10 |   2.8308 | cap4_acc80_repeg0.0 | INC | DEC | C0  | C2  | C1  | C3  |
|    11 |   3.0784 | cap4_acc80_repeg0.5 | C1  | INC | C3  | C2  | DEC | C0  |
|    12 |   3.2508 | cap4_acc80_repeg0.5 | INC | DEC | C0  | C1  | C2  | C3  |
|    13 |   3.2848 | cap4_acc80_repeg0.5 | INC | C1  | C0  | DEC | C2  | C3  |
|    14 |   3.4324 | cap4_acc80_repeg0.0 | DEC | INC | C2  | C3  | C1  | C0  |
|    15 |   4.2000 | cap4_acc80_repeg0.2 | INC | DEC | C0  | C2  | C1  | C3  |
|    16 |   4.2154 | cap4_acc80_repeg0.1 | INC | DEC | C0  | C2  | C1  | C3  |
|    17 |   4.2319 | cap4_acc80_repeg0.2 | DEC | INC | C3  | C1  | C2  | C0  |
|    18 |   4.4444 | symbolic_cap4       | INC | DEC | C3  | C0  | C1  | C2  |
|    19 |   4.4444 | symbolic_cap4       | DEC | INC | C0  | C3  | C2  | C1  |
|    20 |   4.7092 | cap4_acc80_repeg0.1 | DEC | INC | C2  | C3  | C1  | C0  |


### DC-3 · Model Quality Affects Discrimination Sharpness

acc70 vs acc80 models on incremental wiring (cap3 → cap4).  
Lower quality = wider, flatter deficit landscape = more contenders after cliff pruning.  
acc70 recovers the correct cap4 wiring at stage 2, but with 3.5× higher deficit than acc80.

In [12]:
exhibit('cap4_incremental_topk')

# Stage 1 — cap3 departure  (cliff pruning)

Contenders surviving to stage 2: **14**
- **cap4_ghost**: 2
- **cap4_acc70**: 8
- **cap4_acc80**: 4

| Place |  Deficit | Model                         | INC | DEC | C0  | C1  | C2  |
| ----- | -------- | ----------------------------- | --- | --- | --- | --- | --- |
|     1 |   0.0000 | cap4_ghost | INC | DEC | C0  | C1  | C2  |
|     2 |   0.0048 | cap4_acc70     | INC | DEC | C1  | C2  | C3  |
|     3 |   0.0136 | cap4_ghost | DEC | INC | C2  | C1  | C0  |
|     4 |   0.0263 | cap4_acc70     | DEC | INC | C2  | C1  | C0  |
|     5 |   0.0577 | cap4_acc80     | DEC | INC | C2  | C1  | C0  |
|     6 |   0.0661 | cap4_acc70     | INC | DEC | C0  | C2  | C3  |

# Stage 2 — cap4 extension  (S1 Base = stage-1 deficit)

| Place |  Deficit |  S1 Base | Model                     | INC | DEC | C0  | C1  | C2  | C3  |
| ----- | -------- | -------- | ------------------------- | --- | --- | --- | --- | --- | --- |
|     1 |   0.0400 |   0.3795 | cap4_acc80 | INC | DEC | C0  | C1  | C2  | C3  |
|     2 |   0.1386 |   0.1386 | cap4_acc70 | INC | DEC | C0  | C1  | C2  | C3  |
|     3 |   0.6548 |   0.6773 | cap4_acc70 | DEC | INC | C3  | C2  | C1  | C0  |
|     4 |   2.9332 |   0.1386 | cap4_acc70 | INC | DEC | C0  | C1  | C3  | C2  |
|     5 |   3.1905 |   0.5072 | cap4_acc80 | INC | DEC | C0  | C1  | C3  | C2  |
|     6 |   5.7268 |   0.6601 | cap4_acc70 | DEC | INC | C3  | C2  | C0  | C1  |


### DC-4 · Ghost Replication at Cap5 Scale

The ghost result from §4 (deficit gap of 7.03 at cap4) replicates at cap5. The cap5 ghost — trained only to cap4 — collapses on the cap5 world (deficit 2.89) while both native cap5 models score below 0.04. The C4 slot exists architecturally; the representation does not.

In [ ]:
exhibit('cap5_ghost_games')

### DC-5 · Scorer Robustness: Ghost Failure

The ghost failure ordering holds under three scoring functions: deficit (hinge), tiered (stepped penalty), and cross-entropy (NLL). Ghost canonical ranks last in all three; native canonical ranks first among trained models in all three. The failure is structural, not an artifact of the deficit function's shape.

In [ ]:
exhibit('cap4_ghost_games_scorers')

### DC-6 · Scorer Robustness: Succession Crossing

The succession rank crossing — ghost leads stage 1, native leads stage 2 — is present under all three scorers. The stage-2 outcome is unchanged whether the scorer is a hinge loss, a step function, or an unbounded log penalty. The crossing is in the data, not in the scoring function.

In [ ]:
exhibit('cap4_succession_scorers')